# Use a vector database for storage and retrieval

In the [last notebook](11-similarity-embeddings.ipynb) you have seen how
embeddings can be calculated and used for retrieving data. We saved
the embeddings as `.npy` files so that we do not have to calculate them
again.

Retrieval worked by calculating the similarity of the question to all
the answers. In a scenario with just a few thousand documents, this works
well. However, as the number of documents increases, we have to find a more
scalable solution. This can be achieved with a vector database.

The vector database is used for storing the document vectors and for
performing a *similarity search*. In this notebook, we use
[usearch](https://github.com/unum-cloud/usearch) as a vector database.
It is a lightweight solution but explains the concept very well.

## Load data (from previous notebook)

In [1]:
import json
with open("sentences.json") as f:
    sentences = json.load(f)

In [2]:
len(sentences)

18342

In [3]:
import numpy as np
with open("sentences-arctic.npy", "rb") as f:
    sembeddings = np.load(f)

In [4]:
sembeddings.shape

(18342, 1024)

## Use usearch vector DB

In [5]:
from usearch.index import Index, MetricKind

# create an index with the correct number of dimensions
index = Index(ndim=sembeddings.shape[1], metric='cos')

add all vectors to the datbase

In [6]:
%%time
index.add(list(range(len(sembeddings))), sembeddings)

CPU times: user 7.89 s, sys: 44.6 ms, total: 7.93 s
Wall time: 620 ms


array([    0,     1,     2, ..., 18339, 18340, 18341],
      shape=(18342,), dtype=uint64)

In [7]:
index.save("sentences-arctic.usearch")

In [8]:
# need model for calculating new embeddings
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('Snowflake/snowflake-arctic-embed-l-v2.0')

In [9]:
import pandas as pd
from usearch.index import MetricKind
def search(query, index, sentences, model, query_prompt_name=None, top=20):
    # code query to restrict search space
    question_embedding = model.encode(query, normalize_embeddings=True, prompt_name=query_prompt_name)
    
    # search vector database
    hits = index.search(question_embedding, top, MetricKind.Cos)
    
    # Return as dataframe, note that distance and score are different metrics!
    return pd.DataFrame([{ "id": r.key, 
                           "text": sentences[r.key], 
                           "score": 1-r.distance } for r in hits] )

In [10]:
pd.set_option('display.max_colwidth', 0)

In [11]:
search("Is the climate crisis worse for poorer countries?", index, sentences, model, query_prompt_name="query")

,id,text,score
0,2560,"Despite having contributed the least to climate change, it is the poorest and most vulnerable parts of the world that suffer the most devastating consequences.",0.613479
1,18170,More than half of the world’s top 50 most climate-vulnerable countries are home to 40 per cent of people living in extreme poverty.,0.590153
2,13547,"Developing countries have made progress in reducing carbon emissions, but we continue to be the most affected by climate disasters.",0.582720
3,15089,"In the end, the most affected are always the poorest countries and peoples of the world, who are suffering from inflation, food shortages and high fuel prices.",0.570021
4,7976,"The effects of climate change are causing suffering to the most vulnerable communities, especially small island developing States, least developed countries and those affected by conflict.",0.565402
5,15210,"Developing countries, such as Cote d’Ivoire, which are only marginally responsible for climate change, are disproportionately affected and are suffering the most from its consequences.",0.561096
6,8362,"Poor, vulnerable, climate-distressed and resource-challenged developing countries are absolutely fed up and insulted by the unfulfilled perennial promises of the developed world on climate financing.",0.557180
7,5637,"Developing countries, particularly least developed countries, are currently the most vulnerable to the severe consequences of climate change, natural disasters and diseases.",0.556397
8,17543,"These crises are hitting hardest those who are least responsible for their creation — vulnerable populations, women and children and the world’s poorest peoples.",0.556193
9,15376,"It is also no secret that those who are least responsible for climate change are the ones suffering the most from its effects, particularly small island developing States.",0.555936
